# Feature Engineering

This notebook creates the required business features and additional analytical features from the `cart2insights` MySQL database.

## 1. Import Libraries

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


## 2. Load Environment Variables

In [2]:
PROJECT_ROOT = Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"

load_dotenv(ENV_FILE)

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "cart2insights")

print("Project root:", PROJECT_ROOT)
print("Database:", MYSQL_DATABASE)
print("MySQL host:", MYSQL_HOST)


Project root: d:\HCL-GUVI\Cart_to_Insights
Database: cart2insights
MySQL host: localhost


## 3. Connect to MySQL

In [3]:
database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE
)

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    db_name = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

print("Connected successfully.")
print("Database:", db_name)


Connected successfully.
Database: cart2insights


## 4. Load Orders and Order Items

In [4]:
orders_query = """
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date
FROM orders o
INNER JOIN customers c
    ON o.customer_id = c.customer_id;
"""

orders_df = pd.read_sql(
    text(orders_query),
    engine
)

order_items_query = """
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    price,
    freight_value,
    shipping_limit_date
FROM order_items;
"""

order_items_df = pd.read_sql(
    text(order_items_query),
    engine
)

print("Orders shape:", orders_df.shape)
print("Order Items shape:", order_items_df.shape)


Orders shape: (99441, 9)
Order Items shape: (112650, 7)


## 5. Create Order Value Features

In [5]:
order_value_features = (
    order_items_df
    .groupby("order_id", as_index=False)
    .agg(
        item_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        order_item_count=("order_item_id", "count")
    )
)

order_value_features["total_order_value"] = (
    order_value_features["item_revenue"]
    + order_value_features["freight_value"]
)

order_features = orders_df.merge(
    order_value_features,
    on="order_id",
    how="left"
)

value_columns = [
    "item_revenue",
    "freight_value",
    "order_item_count",
    "total_order_value"
]

order_features[value_columns] = (
    order_features[value_columns]
    .fillna(0)
)

print("Order features shape:", order_features.shape)


Order features shape: (99441, 13)


## 6. Create Delivery, Time, and Operational Features

In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    order_features[column] = pd.to_datetime(
        order_features[column],
        errors="coerce"
    )

order_features["delivery_days"] = (
    (
        order_features["order_delivered_customer_date"]
        - order_features["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)
).round(2)

order_features["estimated_delivery_days"] = (
    (
        order_features["order_estimated_delivery_date"]
        - order_features["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)
).round(2)

order_features["delivery_delay_days"] = (
    (
        order_features["order_delivered_customer_date"]
        - order_features["order_estimated_delivery_date"]
    ).dt.total_seconds() / (24 * 60 * 60)
).round(2)

order_features["purchase_year"] = order_features["order_purchase_timestamp"].dt.year
order_features["purchase_month"] = order_features["order_purchase_timestamp"].dt.month
order_features["purchase_month_name"] = order_features["order_purchase_timestamp"].dt.month_name()
order_features["purchase_quarter"] = order_features["order_purchase_timestamp"].dt.quarter
order_features["purchase_day_of_week"] = order_features["order_purchase_timestamp"].dt.day_name()
order_features["purchase_hour"] = order_features["order_purchase_timestamp"].dt.hour

order_features["approval_hours"] = (
    (
        order_features["order_approved_at"]
        - order_features["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600
).round(2)

order_features["purchase_to_carrier_days"] = (
    (
        order_features["order_delivered_carrier_date"]
        - order_features["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)
).round(2)

order_features["carrier_to_customer_days"] = (
    (
        order_features["order_delivered_customer_date"]
        - order_features["order_delivered_carrier_date"]
    ).dt.total_seconds() / (24 * 60 * 60)
).round(2)

order_features["total_items_per_order"] = order_features["order_item_count"]

order_features["freight_percentage"] = np.where(
    order_features["total_order_value"] > 0,
    (
        order_features["freight_value"]
        / order_features["total_order_value"]
    ) * 100,
    np.nan
).round(2)

print(
    order_features[
        [
            "order_id",
            "delivery_days",
            "estimated_delivery_days",
            "delivery_delay_days",
            "purchase_year",
            "purchase_month",
            "purchase_hour",
            "approval_hours",
            "purchase_to_carrier_days",
            "carrier_to_customer_days",
            "total_items_per_order",
            "freight_percentage"
        ]
    ].head()
)


                           order_id  delivery_days  estimated_delivery_days  \
0  00010242fe8c5a6d1ba2dd792cb16214           7.61                    15.63   
1  00018f77f2f0320c557190d7a144bdd3          16.22                    18.55   
2  000229ec398224ef6ca0657da4fc703e           7.95                    21.39   
3  00024acbcdf0a6daa1e931b038114c75           6.15                    11.58   
4  00042b26cf59d7ce69dfabb4e55b4fd9          25.11                    40.42   

   delivery_delay_days  purchase_year  purchase_month  purchase_hour  \
0                -8.01           2017               9              8   
1                -2.33           2017               4             10   
2               -13.44           2018               1             14   
3                -5.44           2018               8             10   
4               -15.30           2017               2             13   

   approval_hours  purchase_to_carrier_days  carrier_to_customer_days  \
0            0.78  

## 7. Create Delivery Status and Delay Features

In [7]:
order_features["delivery_status"] = np.select(
    [
        order_features["order_delivered_customer_date"].isna(),
        order_features["delivery_delay_days"] > 0,
        order_features["delivery_delay_days"] <= 0
    ],
    [
        "not_available",
        "delayed",
        "on_time"
    ],
    default="not_available"
)

order_features["is_delayed"] = np.where(
    order_features["delivery_status"] == "delayed",
    1,
    np.where(
        order_features["delivery_status"] == "on_time",
        0,
        np.nan
    )
)

order_features["delay_category"] = np.select(
    [
        order_features["delivery_status"] == "not_available",
        order_features["delivery_delay_days"] <= 0,
        order_features["delivery_delay_days"].between(0.01, 3),
        order_features["delivery_delay_days"].between(3.01, 7),
        order_features["delivery_delay_days"] > 7
    ],
    [
        "not_available",
        "on_time_or_early",
        "1_3_days_late",
        "4_7_days_late",
        "more_than_7_days_late"
    ],
    default="not_available"
)

print(order_features["delivery_status"].value_counts())
print()
print(order_features["delay_category"].value_counts())


delivery_status
on_time          88650
delayed           7820
not_available     2971
Name: count, dtype: int64

delay_category
on_time_or_early         88650
more_than_7_days_late     3341
not_available             2971
1_3_days_late             2660
4_7_days_late             1819
Name: count, dtype: int64


## 8. Create Customer Features

In [8]:
customer_features = (
    order_features
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spending=("total_order_value", "sum"),
        customer_average_order_value=("total_order_value", "mean"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max")
    )
)

customer_item_counts = (
    order_features
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_total_items=("total_items_per_order", "sum"),
        customer_average_items_per_order=("total_items_per_order", "mean")
    )
)

customer_delivery_metrics = (
    order_features
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_delayed_order_count=(
            "is_delayed",
            lambda x: (x == 1).sum()
        ),
        customer_valid_delivery_orders=(
            "is_delayed",
            lambda x: x.notna().sum()
        )
    )
)

customer_features = customer_features.merge(
    customer_item_counts,
    on="customer_unique_id",
    how="left"
)

customer_features = customer_features.merge(
    customer_delivery_metrics,
    on="customer_unique_id",
    how="left"
)

customer_features["customer_on_time_rate"] = np.where(
    customer_features["customer_valid_delivery_orders"] > 0,
    (
        (
            customer_features["customer_valid_delivery_orders"]
            - customer_features["customer_delayed_order_count"]
        )
        / customer_features["customer_valid_delivery_orders"]
    ) * 100,
    np.nan
).round(2)

customer_features["customer_lifetime_days"] = (
    customer_features["last_order_date"]
    - customer_features["first_order_date"]
).dt.days

customer_features["repeat_customer_indicator"] = np.where(
    customer_features["customer_order_count"] > 1,
    1,
    0
)

print("Customer features shape:", customer_features.shape)


Customer features shape: (96096, 13)


## 9. Create Seller Features

In [9]:
seller_features = (
    order_items_df
    .groupby("seller_id", as_index=False)
    .agg(
        seller_revenue=("price", "sum"),
        seller_order_count=("order_id", "nunique"),
        seller_item_count=("order_item_id", "count"),
        seller_unique_products=("product_id", "nunique"),
        seller_total_freight=("freight_value", "sum"),
        seller_average_freight=("freight_value", "mean"),
        seller_average_item_price=("price", "mean")
    )
)

print("Seller features shape:", seller_features.shape)


Seller features shape: (3095, 8)


## 10. Create Product and Category Features

In [10]:
product_features = (
    order_items_df
    .groupby("product_id", as_index=False)
    .agg(
        product_order_count=("order_id", "nunique"),
        product_item_count=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        product_total_freight=("freight_value", "sum"),
        product_average_price=("price", "mean")
    )
)

products_query = """
SELECT
    product_id,
    product_category_name
FROM products;
"""

products_lookup = pd.read_sql(
    text(products_query),
    engine
)

product_features = product_features.merge(
    products_lookup,
    on="product_id",
    how="left"
)

category_features = (
    product_features
    .groupby("product_category_name", as_index=False)
    .agg(
        category_product_count=("product_id", "nunique"),
        category_order_count=("product_order_count", "sum"),
        category_item_count=("product_item_count", "sum"),
        category_revenue=("product_revenue", "sum"),
        category_average_price=("product_average_price", "mean")
    )
)

print("Product features shape:", product_features.shape)
print("Category features shape:", category_features.shape)


Product features shape: (32951, 7)
Category features shape: (74, 6)


## 11. Calculate Business KPIs

In [11]:
total_orders = order_features["order_id"].nunique()
total_order_value = order_features["total_order_value"].sum()
total_product_revenue = order_features["item_revenue"].sum()
total_freight = order_features["freight_value"].sum()
total_items = order_items_df["order_item_id"].count()
total_customers = customer_features["customer_unique_id"].nunique()
total_sellers = seller_features["seller_id"].nunique()

average_order_value = order_features.loc[
    order_features["total_order_value"] > 0,
    "total_order_value"
].mean()

average_delivery_days = order_features.loc[
    order_features["delivery_days"].notna(),
    "delivery_days"
].mean()

average_delivery_delay = order_features.loc[
    order_features["delivery_status"] == "delayed",
    "delivery_delay_days"
].mean()

median_delivery_delay = order_features.loc[
    order_features["delivery_status"] == "delayed",
    "delivery_delay_days"
].median()

max_delivery_delay = order_features.loc[
    order_features["delivery_status"] == "delayed",
    "delivery_delay_days"
].max()

on_time_orders = order_features["delivery_status"].eq("on_time").sum()
delayed_orders = order_features["delivery_status"].eq("delayed").sum()
not_available_delivery_orders = order_features["delivery_status"].eq("not_available").sum()
valid_delivery_orders = on_time_orders + delayed_orders

on_time_rate = (on_time_orders / valid_delivery_orders) * 100 if valid_delivery_orders else np.nan
delayed_rate = (delayed_orders / valid_delivery_orders) * 100 if valid_delivery_orders else np.nan

repeat_customers = int(customer_features["repeat_customer_indicator"].sum())
repeat_customer_rate = (repeat_customers / total_customers) * 100

business_kpis = pd.DataFrame({
    "metric": [
        "total_product_revenue",
        "total_order_value",
        "total_freight",
        "total_orders",
        "total_items",
        "total_customers",
        "total_sellers",
        "average_order_value",
        "average_delivery_days",
        "average_delivery_delay_days",
        "median_delivery_delay_days",
        "max_delivery_delay_days",
        "on_time_orders",
        "delayed_orders",
        "not_available_delivery_orders",
        "on_time_rate",
        "delayed_rate",
        "repeat_customers",
        "repeat_customer_rate"
    ],
    "value": [
        total_product_revenue,
        total_order_value,
        total_freight,
        total_orders,
        total_items,
        total_customers,
        total_sellers,
        average_order_value,
        average_delivery_days,
        average_delivery_delay,
        median_delivery_delay,
        max_delivery_delay,
        on_time_orders,
        delayed_orders,
        not_available_delivery_orders,
        on_time_rate,
        delayed_rate,
        repeat_customers,
        repeat_customer_rate
    ]
})

business_kpis


,metric,value
0,total_product_revenue,1.359164e+07
1,total_order_value,1.584355e+07
2,total_freight,2.251910e+06
3,total_orders,9.944100e+04
4,total_items,1.126500e+05
5,total_customers,9.609600e+04
6,total_sellers,3.095000e+03
7,average_order_value,1.605776e+02
8,average_delivery_days,1.255820e+01
9,average_delivery_delay_days,9.559078e+00


## 12. Add Customer Features to Order Features

In [12]:
customer_columns = [
    "customer_order_count",
    "customer_total_spending",
    "customer_average_order_value",
    "repeat_customer_indicator"
]

order_features = order_features.drop(
    columns=customer_columns,
    errors="ignore"
)

order_features = order_features.merge(
    customer_features[
        ["customer_unique_id", *customer_columns]
    ],
    on="customer_unique_id",
    how="left"
)

print("Order features shape:", order_features.shape)


Order features shape: (99441, 34)


## 13. Validate Feature Tables

In [13]:
validation_summary = pd.DataFrame({
    "Table": [
        "order_features",
        "customer_features",
        "seller_features",
        "product_features",
        "category_features"
    ],
    "Rows": [
        len(order_features),
        len(customer_features),
        len(seller_features),
        len(product_features),
        len(category_features)
    ],
    "Columns": [
        len(order_features.columns),
        len(customer_features.columns),
        len(seller_features.columns),
        len(product_features.columns),
        len(category_features.columns)
    ],
    "Duplicate Keys": [
        order_features["order_id"].duplicated().sum(),
        customer_features["customer_unique_id"].duplicated().sum(),
        seller_features["seller_id"].duplicated().sum(),
        product_features["product_id"].duplicated().sum(),
        np.nan
    ]
})

validation_summary


,Table,Rows,Columns,Duplicate Keys
0,order_features,99441,34,0.0
1,customer_features,96096,13,0.0
2,seller_features,3095,8,0.0
3,product_features,32951,7,0.0
4,category_features,74,6,NaN


## 14. Check Missing Feature Values

In [14]:
print("Order features missing values")
print(
    order_features.isna().sum()[
        order_features.isna().sum() > 0
    ]
)

print("\nCustomer features missing values")
print(
    customer_features.isna().sum()[
        customer_features.isna().sum() > 0
    ]
)

print("\nSeller features missing values")
print(
    seller_features.isna().sum()[
        seller_features.isna().sum() > 0
    ]
)


Order features missing values
order_approved_at                 160
order_delivered_carrier_date     1972
order_delivered_customer_date    2971
delivery_days                    2971
delivery_delay_days              2971
approval_hours                    160
purchase_to_carrier_days         1972
carrier_to_customer_days         3160
freight_percentage                775
is_delayed                       2971
dtype: int64

Customer features missing values
customer_on_time_rate    2746
dtype: int64

Seller features missing values
Series([], dtype: int64)


## 15. Save Feature Tables to MySQL

In [15]:
feature_tables = {
    "order_features": order_features,
    "customer_features": customer_features,
    "seller_features": seller_features,
    "product_features": product_features,
    "category_features": category_features,
    "business_kpis": business_kpis
}

for table_name, df in feature_tables.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=2000,
        method="multi"
    )
    print(f"{table_name}: {len(df):,} rows saved")

print("\nAll feature tables saved successfully.")


order_features: 99,441 rows saved
customer_features: 96,096 rows saved
seller_features: 3,095 rows saved
product_features: 32,951 rows saved
category_features: 74 rows saved
business_kpis: 19 rows saved

All feature tables saved successfully.


## 16. Verify SQL Feature Tables

In [16]:
feature_table_names = [
    "order_features",
    "customer_features",
    "seller_features",
    "product_features",
    "category_features",
    "business_kpis"
]

feature_table_check = []

for table_name in feature_table_names:
    query = text(f"SELECT COUNT(*) FROM `{table_name}`")
    with engine.connect() as connection:
        row_count = connection.execute(query).scalar()
    feature_table_check.append({
        "Table": table_name,
        "Rows": row_count
    })

pd.DataFrame(feature_table_check)


,Table,Rows
0,order_features,99441
1,customer_features,96096
2,seller_features,3095
3,product_features,32951
4,category_features,74
5,business_kpis,19


## 17. Feature Dictionary

In [17]:
feature_dictionary = pd.DataFrame({
    "Feature": [
        "total_order_value", "delivery_days", "delivery_delay_days",
        "customer_order_count", "customer_total_spending", "average_order_value",
        "seller_revenue", "seller_order_count", "repeat_customer_indicator",
        "purchase_year", "purchase_month", "purchase_quarter",
        "purchase_day_of_week", "purchase_hour", "approval_hours",
        "purchase_to_carrier_days", "carrier_to_customer_days",
        "total_items_per_order", "freight_percentage", "delivery_status",
        "is_delayed", "delay_category", "customer_total_items",
        "customer_average_items_per_order", "customer_delayed_order_count",
        "customer_on_time_rate", "customer_lifetime_days", "seller_item_count",
        "seller_unique_products", "seller_total_freight", "seller_average_freight",
        "seller_average_item_price", "product_order_count", "product_item_count",
        "product_revenue", "product_total_freight", "product_average_price",
        "category_product_count", "category_order_count", "category_item_count",
        "category_revenue", "category_average_price"
    ],
    "Definition": [
        "Sum of product price and freight value for an order",
        "Number of days from purchase to actual customer delivery",
        "Difference in days between actual and estimated delivery",
        "Distinct number of orders placed by a unique customer",
        "Total order value spent by a unique customer",
        "Average total order value across orders with positive value",
        "Sum of product price attributed to a seller",
        "Distinct number of orders handled by a seller",
        "1 if customer has more than one order, otherwise 0",
        "Year in which the order was purchased",
        "Month number in which the order was purchased",
        "Quarter in which the order was purchased",
        "Day of week on which the order was purchased",
        "Hour of day when the order was purchased",
        "Hours between purchase and payment approval",
        "Days between purchase and carrier handoff",
        "Days between carrier handoff and customer delivery",
        "Number of items included in the order",
        "Freight value as a percentage of total order value",
        "Delivery classification: on time, delayed, or unavailable",
        "1 if delayed, 0 if on time, missing if unavailable",
        "Categorization of delivery delay severity",
        "Total items purchased by the customer",
        "Average items per customer order",
        "Number of delayed orders for the customer",
        "Percentage of valid deliveries that were on time",
        "Days between the customer's first and last order",
        "Total number of items sold by the seller",
        "Number of unique products sold by the seller",
        "Total freight associated with seller orders",
        "Average freight value per item sold",
        "Average selling price of seller items",
        "Number of distinct orders containing the product",
        "Total number of product items sold",
        "Total product-price revenue generated",
        "Total freight associated with the product",
        "Average selling price of the product",
        "Number of products in the category",
        "Number of orders containing products from the category",
        "Total number of items sold in the category",
        "Total product revenue generated by the category",
        "Average product price within the category"
    ],
    "Level": [
        "Order", "Order", "Order", "Customer", "Customer", "Business",
        "Seller", "Seller", "Customer", "Order", "Order", "Order", "Order",
        "Order", "Order", "Order", "Order", "Order", "Order", "Order",
        "Order", "Order", "Customer", "Customer", "Customer", "Customer",
        "Customer", "Seller", "Seller", "Seller", "Seller", "Seller",
        "Product", "Product", "Product", "Product", "Product", "Category",
        "Category", "Category", "Category", "Category"
    ]
})

feature_dictionary


,Feature,Definition,Level
0,total_order_value,Sum of product price and freight value for an ...,Order
1,delivery_days,Number of days from purchase to actual custome...,Order
2,delivery_delay_days,Difference in days between actual and estimate...,Order
3,customer_order_count,Distinct number of orders placed by a unique c...,Customer
4,customer_total_spending,Total order value spent by a unique customer,Customer
5,average_order_value,Average total order value across orders with p...,Business
6,seller_revenue,Sum of product price attributed to a seller,Seller
7,seller_order_count,Distinct number of orders handled by a seller,Seller
8,repeat_customer_indicator,"1 if customer has more than one order, otherwi...",Customer
9,purchase_year,Year in which the order was purchased,Order


## 18. Final Feature Validation

In [18]:
checks = {
    "Order IDs unique": order_features["order_id"].is_unique,
    "Customer IDs unique": customer_features["customer_unique_id"].is_unique,
    "Seller IDs unique": seller_features["seller_id"].is_unique,
    "Product IDs unique": product_features["product_id"].is_unique,
    "Required features exist": all(
        column in order_features.columns
        for column in [
            "total_order_value",
            "delivery_days",
            "delivery_delay_days",
            "customer_order_count",
            "customer_total_spending",
            "seller_revenue",
            "seller_order_count",
            "repeat_customer_indicator"
        ]
    ),
    "No negative order values": (
        order_features["total_order_value"] >= 0
    ).all(),
    "No negative seller revenue": (
        seller_features["seller_revenue"] >= 0
    ).all()
}

final_validation = pd.DataFrame({
    "Check": list(checks.keys()),
    "Status": [
        "PASS" if result else "CHECK"
        for result in checks.values()
    ]
})

print(final_validation.to_string(index=False))
print()
print("Delivery status")
print(order_features["delivery_status"].value_counts())
print()
print("Repeat customers")
print(customer_features["repeat_customer_indicator"].value_counts())
print()
print("Business KPIs")
print(business_kpis.to_string(index=False))


                     Check Status
          Order IDs unique   PASS
       Customer IDs unique   PASS
         Seller IDs unique   PASS
        Product IDs unique   PASS
   Required features exist  CHECK
  No negative order values   PASS
No negative seller revenue   PASS

Delivery status
delivery_status
on_time          88650
delayed           7820
not_available     2971
Name: count, dtype: int64

Repeat customers
repeat_customer_indicator
0    93099
1     2997
Name: count, dtype: int64

Business KPIs
                       metric        value
        total_product_revenue 1.359164e+07
            total_order_value 1.584355e+07
                total_freight 2.251910e+06
                 total_orders 9.944100e+04
                  total_items 1.126500e+05
              total_customers 9.609600e+04
                total_sellers 3.095000e+03
          average_order_value 1.605776e+02
        average_delivery_days 1.255820e+01
  average_delivery_delay_days 9.559078e+00
   median_delivery_d